In [21]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh
import math


def WS_sparse_pauli_op(N, x, lam, l0, m_lat, g):
    """
    Construct the SparsePauliOp for

        W_S =  (x/2) * sum_{n=0}^{N-2} (X_n X_{n+1} + Y_n Y_{n+1})
             + (1/2) * sum_{n=0}^{N-2} sum_{k=n+1}^{N-1} (N - k - 1 + lam) * Z_n Z_k
             + sum_{n=0}^{N-2} ( N/4 - (1/2) * floor(n/2) + l0*(N - n - 1) ) * Z_n
             + (m_lat/g) * sqrt(x) * sum_{n=0}^{N-1} (-1)^n * Z_n
             + l0**2 * (N - 1) + (1/2) * l0 * N + (1/8) * N**2 + (lam/4) * N

    Args:
        N (int): number of qubits (sites).
        x (float)
        lam (float): lambda
        l0 (float): ell_0
        m_lat (float)
        g (float)

    Returns:
        SparsePauliOp on N qubits.
    """
    terms = []

    def add_1q(op, i, coeff):
        # Qiskit labels are little-endian: qubit 0 is rightmost
        s = ['I'] * N
        s[N - 1 - i] = op
        terms.append((''.join(s), complex(coeff)))

    def add_2q(op, i, j, coeff):
        s = ['I'] * N
        s[N - 1 - i] = op
        s[N - 1 - j] = op
        terms.append((''.join(s), complex(coeff)))

    # (x/2) * sum (X_i X_{i+1} + Y_i Y_{i+1})
    for n in range(N - 1):
        add_2q('X', n, n + 1, x / 2.0)
        add_2q('Y', n, n + 1, x / 2.0)

    # (1/2) * sum_{n<k} (N - k - 1 + lam) * Z_n Z_k
    for n in range(N - 1):
        for k in range(n + 1, N):
            coeff = 0.5 * (N - k - 1 + lam)
            add_2q('Z', n, k, coeff)

    # sum_{n=0}^{N-2} (N/4 - 1/2 floor(n/2) + l0*(N - n - 1)) * Z_n
    for n in range(N - 1):
        coeff = (N / 4.0) - 0.5 * math.ceil(n / 2) + l0 * (N - n - 1)
        add_1q('Z', n, coeff)

    # (m_lat/g) * sqrt(x) * sum_{n=0}^{N-1} (-1)^n * Z_n
    pref = (m_lat / g) * math.sqrt(x)
    for n in range(N):
        add_1q('Z', n, pref * ((-1) ** n))

    # Constant (identity) term
    const = (l0 ** 2) * (N - 1) + 0.5 * l0 * N + 0.125 * (N ** 2) + (lam / 4.0) * N
    terms.append(('I' * N, complex(const)))

    return SparsePauliOp.from_list(terms).simplify()

In [8]:
m = 0.5
g = 0.3
mat = WS_sparse_pauli_op(8, 1. / (g**2), 100., 0., m, g).to_matrix(sparse=True)
eigen_values, eigen_vectors = eigsh(mat, k=254, which="SM")
sorted(eigen_values / 8)
np.abs(eigen_vectors.imag).sum(axis=1)

array([2.37143096e-14, 1.46346255e+00, 1.09139688e+00, 1.74104424e+00,
       1.21681705e+00, 1.83213301e+00, 2.37512650e+00, 2.58867554e+00,
       1.09933538e+00, 2.67990384e+00, 1.83877442e+00, 2.95155831e+00,
       2.61831208e+00, 3.69152187e+00, 3.13617175e+00, 2.34782326e+00,
       1.36545359e+00, 2.60829269e+00, 2.23522325e+00, 3.71479835e+00,
       2.65609126e+00, 3.49928960e+00, 3.90831702e+00, 2.18883861e+00,
       2.48135449e+00, 3.84343069e+00, 3.69096485e+00, 3.81957988e+00,
       3.58711203e+00, 3.30523506e+00, 4.04633241e+00, 1.86129272e+00,
       1.02352291e+00, 2.90825529e+00, 2.51778835e+00, 3.41985806e+00,
       2.94451089e+00, 4.16599214e+00, 3.91494624e+00, 3.81957988e+00,
       2.22523104e+00, 4.14862553e+00, 3.21183876e+00, 2.91125679e+00,
       3.96112423e+00, 4.43043770e+00, 3.53205876e+00, 2.78872539e+00,
       2.84861794e+00, 3.84423503e+00, 3.82443493e+00, 3.12192748e+00,
       4.55206011e+00, 3.78199756e+00, 4.26707623e+00, 3.24358503e+00,
      

In [3]:
eigen_vectors: np.ndarray
np.abs(eigen_vectors.imag).sum(axis=0)

array([1.57152396, 0.54546268, 1.46565944, 1.39026888, 0.88956796,
       1.45130986, 2.22307133, 0.79673999, 1.43056283, 0.91649361,
       0.29526501, 0.06262132, 0.45459031, 1.7033163 , 2.04030088,
       2.29877404, 2.49866033, 3.69516866, 1.34002081, 3.07978967,
       3.66601571, 1.75150157, 4.19198848, 3.16184349, 1.74712963,
       0.13722244, 2.83911694, 2.64123511, 1.76386086, 0.99066228,
       2.88272118, 3.82327356, 3.85045438, 2.35129316, 2.03665953,
       3.40539995, 3.68402822, 1.17512942, 3.74138703, 1.64739126,
       2.91208308, 1.61347771, 2.81966637, 1.73186446, 0.15969642,
       1.65309078, 2.10272509, 3.17621856, 3.74953301, 3.4610272 ,
       1.02226273, 4.02407036, 2.95756517, 3.4972433 , 3.49369466,
       2.33059425, 3.76606877, 3.1156031 , 2.29272813, 3.07838105,
       2.01041137, 1.87187633, 3.4261867 , 1.80132218, 2.07803002,
       1.15685856, 0.8715135 , 2.76763141, 3.48968916, 3.27614451,
       1.30879622, 0.8524287 , 2.38224671, 0.45515158, 3.32228

In [50]:
from hamiltonian.free_wilson import FreeWilson2D
from hamiltonian.base import HamiltonianType

hamiltonian = FreeWilson2D(2,2, -3.5, 1)
print(hamiltonian.size_of_zero_charge_sector())
h_operator = hamiltonian.hamiltonian_op(HamiltonianType.ZeroChargeProjection)
h_sparse_matrix = h_operator.to_matrix(sparse=True)
print(h_sparse_matrix.shape)
h_sparse_matrix: csr_matrix
print(len(h_sparse_matrix.data), h_sparse_matrix.count_nonzero())
h_sparse_matrix.eliminate_zeros()
print(len(h_sparse_matrix.data), h_sparse_matrix.count_nonzero())
print(h_sparse_matrix)
print(len(h_sparse_matrix.indptr), h_sparse_matrix.indptr)
print(len(h_sparse_matrix.indices), h_sparse_matrix.indices)
temp = h_sparse_matrix.tocoo()
print(temp.row)
print(temp.col)
print(temp.data)


final_dat = []
row_ind = []
col_ind = []
zero_charge_size = hamiltonian.size_of_zero_charge_sector()
my_n = hamiltonian.N_sites

my_ind_map = []
id_temp = 0
for i in range(temp.shape[0]):
    bin_i = np.binary_repr(i, my_n)
    i_ones = bin_i.count("1")
    if i_ones == (my_n // 2):
        my_ind_map.append(id_temp)
        id_temp+=1
    else:
        my_ind_map.append(-1)

for i, j, dat in zip(temp.row, temp.col,temp.data):
    bin_i = np.binary_repr(i, my_n)
    bin_j = np.binary_repr(j, my_n)
    i_ones = bin_i.count("1")
    j_ones = bin_j.count("1")
    if (i_ones == (my_n//2)) and (j_ones == (my_n//2)):
        row_ind.append(my_ind_map[i])
        col_ind.append(my_ind_map[j])
        final_dat.append(dat)


new_h = csr_matrix((final_dat, (row_ind, col_ind)), shape=(zero_charge_size, zero_charge_size))
print(new_h, new_h.shape)


np.abs(h_sparse_matrix.imag).sum()
# in full hamiltonian of FreeWilson2D:
# temp = (c + c.adjoint()).to_matrix(sparse=True)
# print(np.abs(temp.imag).sum())

70
(256, 256)
2304 354
354 354
<Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 354 stored elements and shape (256, 256)>
  Coords	Values
  (15, 23)	(1+0j)
  (15, 43)	(1+0j)
  (15, 77)	(1+0j)
  (15, 142)	(1+0j)
  (23, 15)	(1+0j)
  (23, 29)	(-1+0j)
  (23, 51)	(1+0j)
  (23, 71)	(1+0j)
  (23, 85)	(1+0j)
  (23, 150)	(1+0j)
  (27, 27)	(3+0j)
  (27, 30)	(1+0j)
  (27, 75)	(1+0j)
  (27, 89)	(1+0j)
  (27, 154)	(1+0j)
  (29, 23)	(-1+0j)
  (29, 57)	(-1+0j)
  (29, 77)	(1+0j)
  (29, 156)	(1+0j)
  (30, 27)	(1+0j)
  (30, 30)	(3+0j)
  (30, 58)	(-1+0j)
  (30, 78)	(1+0j)
  (30, 92)	(-1+0j)
  (39, 39)	(-3+0j)
  :	:
  (216, 216)	(3+0j)
  (225, 163)	(-1+0j)
  (225, 177)	(-1+0j)
  (225, 197)	(-1+0j)
  (225, 225)	(-3+0j)
  (225, 228)	(-1+0j)
  (226, 99)	(1+0j)
  (226, 178)	(-1+0j)
  (226, 198)	(-1+0j)
  (226, 232)	(1+0j)
  (228, 101)	(1+0j)
  (228, 166)	(1+0j)
  (228, 180)	(-1+0j)
  (228, 225)	(-1+0j)
  (228, 228)	(-3+0j)
  (232, 105)	(1+0j)
  (232, 170)	(1+0j)
  (232, 184)	(-1+0j)
  (232, 20

np.float64(0.0)

In [117]:
from qiskit_ibm_runtime.estimator import EstimatorV2 as Estimator
from qiskit_aer import AerSimulator
from qiskit.primitives import StatevectorEstimator
from qiskit import generate_preset_pass_manager, QuantumCircuit
from solver.circuits import HardwareAdaptAnsatz10

temp = HardwareAdaptAnsatz10(32)
temp.set_ansatz([0])

backend = AerSimulator(method="statevector")
print(backend.available_devices(), backend.available_methods())

pm = generate_preset_pass_manager(backend=backend)

estimator = Estimator(mode=backend)


pm = generate_preset_pass_manager()

estimator = StatevectorEstimator(default_precision=0)
n_qbits = 29
circ = QuantumCircuit(n_qbits)
for i in range(n_qbits):
    circ.h(i)
circuit = pm.run(circ)

hamiltonian = FreeWilson2D(4,4, -3.5, 1)
h_operator = hamiltonian.hamiltonian_op(HamiltonianType.Full)
h_operator = SparsePauliOp("Z"*n_qbits)
h_operator += SparsePauliOp("Y"*n_qbits)
h_operator

('CPU',) ('automatic', 'statevector', 'density_matrix', 'stabilizer', 'matrix_product_state', 'extended_stabilizer', 'unitary', 'superop')


SparsePauliOp(['ZZZZZZZZZZZZZZZZZZZZZZZZZZZZZ', 'YYYYYYYYYYYYYYYYYYYYYYYYYYYYY'],
              coeffs=[1.+0.j, 1.+0.j])

In [118]:
pub = (circuit, [op.apply_layout(circuit.layout) for op in [h_operator]])
# noinspection PyTypeChecker
job = estimator.run(pubs=[pub])
full_result = job.result()
pub_result = full_result[0]
gradients = pub_result.data.evs
gradients

array([0.])